# PennyLane (Xanadu) — Framework de ML Quântico

## O que é o PennyLane?

**PennyLane** é o framework de *quantum machine learning* (QML) da Xanadu.
Sua característica única: trata circuitos quânticos como **funções Python diferenciáveis**.

Isso significa que você pode calcular gradientes de circuitos quânticos e
**usá-los diretamente em otimizadores** como Adam, SGD ou LBFGS — exatamente
como faz com redes neurais clássicas.

## Arquitetura central

```
  Dados → [ Circuito Quântico ] → Expectação → Loss → Gradiente → Otimizador
               ↑__ parâmetros ajustáveis __↗
```

| Conceito | O que é |
|----------|----------|
| `@qml.qnode` | Transforma uma função Python em circuito quântico executável |
| `qml.device` | Seleciona o backend (simulador local, GPU, hardware real) |
| `qml.grad` | Gradiente via *parameter-shift rule* (exato, não aproximado) |
| `qml.expval` | Valor esperado de um observable — a "saída" do circuito |
| `qml.probs` | Probabilidades de cada estado de base |

## Por que o PennyLane é especial?

```
✅ Gradientes exatos de circuitos quânticos (parameter-shift rule)
✅ Integração nativa com PyTorch, TensorFlow e JAX
✅ Agnóstico de hardware: muda o device, o circuito é o mesmo
✅ Ideal para QML, VQE, QAOA, pesquisa em otimização quântica
✅ Circuitos como camadas em redes neurais (TorchLayer, KerasLayer)

❌ Não é ideal para algoritmos sem parâmetros treináveis (use Qiskit)
❌ Menor controle de scheduling físico (use Cirq para isso)
```

## Exemplo 1 — Bell State e `qml.probs`

O `@qml.qnode` transforma uma função Python comum em um **nó quântico**:
ele é executado no `device` especificado (simulador, GPU, ou hardware real)
e retorna resultados clássicos.

### Observables disponíveis
- `qml.probs(wires)` → probabilidades de cada estado (vetor)
- `qml.expval(op)` → valor esperado ∈ [-1, +1]
- `qml.state()` → vetor de estado completo (statevector)
- `qml.sample(op)` → amostras aleatórias

In [6]:
import pennylane as qml
import numpy as np

# Device: backend de execução
# 'default.qubit' = simulador statevector local (exato, sem ruído)
dev = qml.device("default.qubit", wires=2)

# @qml.qnode: transforma a função em circuito executável
@qml.qnode(dev)
def bell_state():
    qml.Hadamard(wires=0)       # superposição no qubit 0
    qml.CNOT(wires=[0, 1])      # entrelaça com qubit 1
    return qml.probs(wires=[0, 1])  # probabilidades de |00⟩,|01⟩,|10⟩,|11⟩

resultado = bell_state()
estados = ['|00⟩', '|01⟩', '|10⟩', '|11⟩']
print('Bell State — probabilidades:')
for estado, prob in zip(estados, resultado):
    print(f'  {estado}: {prob:.3f}')
# Esperado: |00⟩=0.5, |11⟩=0.5 (entrelaçamento perfeito)

# Visualiza o circuito
print()
print(qml.draw(bell_state)())

Bell State — probabilidades:
  |00⟩: 0.500
  |01⟩: 0.000
  |10⟩: 0.000
  |11⟩: 0.500

0: ──H─╭●─┤ ╭Probs
1: ────╰X─┤ ╰Probs


## Exemplo 2 — Parameter-Shift Rule: Gradientes Exatos

Esta é **a** funcionalidade central do PennyLane.

A **parameter-shift rule** calcula gradientes de circuitos quânticos
de forma **exata** (não aproximada como diferença finita):

$$\frac{\partial f}{\partial \theta} = \frac{f(\theta + \pi/2) - f(\theta - \pi/2)}{2}$$

Isso significa apenas **2 execuções do circuito por parâmetro**.
O resultado é matematicamente idêntico ao `autograd` do PyTorch,
mas para gates quânticos.

In [7]:
import pennylane as qml
import pennylane.numpy as qml_np

dev = qml.device("default.qubit", wires=2)

@qml.qnode(dev)
def circuito_variacional(params):
    qml.RY(params[0], wires=0)   # rotação paramétrica em Y
    qml.RZ(params[1], wires=0)   # rotação paramétrica em Z
    qml.CNOT(wires=[0, 1])       # entrelaçamento
    return qml.expval(qml.PauliZ(0))  # expectação de Z (saída ∈ [-1, +1])

# requires_grad=True: marca os parâmetros como diferenciáveis
params = qml_np.array([0.5, 0.1], requires_grad=True)

# ---- Gradiente exato via parameter-shift rule ----
grad_fn = qml.grad(circuito_variacional)
gradientes = grad_fn(params)
print(f'Valor do circuito em params={params}: {circuito_variacional(params):.4f}')
print(f'Gradientes [dC/dθ₀, dC/dθ₁]: {gradientes}')
print()

# ---- Verificação manual da parameter-shift rule ----
shift = np.pi / 2
params_plus  = qml_np.array([params[0] + shift, params[1]], requires_grad=False)
params_minus = qml_np.array([params[0] - shift, params[1]], requires_grad=False)
grad_manual = (circuito_variacional(params_plus) - circuito_variacional(params_minus)) / 2
print(f'Gradiente manual θ₀: {grad_manual:.6f}')
print(f'Gradiente via qml.grad θ₀: {gradientes[0]:.6f}')
print('✅ São iguais! Parameter-shift rule é exata.')

Valor do circuito em params=[0.5 0.1]: 0.8776
Gradientes [dC/dθ₀, dC/dθ₁]: [-4.79425539e-01 -2.56838940e-18]

Gradiente manual θ₀: -0.479426
Gradiente via qml.grad θ₀: -0.479426
✅ São iguais! Parameter-shift rule é exata.


## Exemplo 3 — Otimização de Circuito Variacional

Aqui treinamos o circuito como se fosse uma rede neural:
minimizando uma função de custo ajustando os parâmetros.

**Objetivo:** encontrar os ângulos que minimizam `⟨Z⟩` → estado aponta para baixo (`|1⟩`).

### Otimizadores disponíveis no PennyLane
| Otimizador | Equivalente clássico |
|------------|----------------------|
| `GradientDescentOptimizer` | SGD |
| `AdamOptimizer` | Adam |
| `NesterovMomentumOptimizer` | SGD + Momentum |
| `RMSPropOptimizer` | RMSProp |
| `QNGOptimizer` | Quantum Natural Gradient (usa métrica de Fisher) |

In [8]:
import pennylane as qml
import pennylane.numpy as qml_np
import numpy as np

dev = qml.device("default.qubit", wires=2)

@qml.qnode(dev)
def circuito_variacional(params):
    qml.RY(params[0], wires=0)
    qml.RZ(params[1], wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.expval(qml.PauliZ(0))

params = qml_np.array([0.5, 0.1], requires_grad=True)

# ---- Gradient Descent ----
print('=== Gradient Descent (SGD) ===')
opt = qml.GradientDescentOptimizer(stepsize=0.4)
params_gd = qml_np.array([0.5, 0.1], requires_grad=True)
for step in range(50):
    params_gd, custo = opt.step_and_cost(circuito_variacional, params_gd)
    if step % 10 == 0:
        print(f'  Step {step:2d}: custo={custo:.4f}, θ=[{params_gd[0]:.3f}, {params_gd[1]:.3f}]')

print(f'  Resultado final: ⟨Z⟩ = {circuito_variacional(params_gd):.4f} (mínimo = -1.0)')

print()

# ---- Adam (converge mais rápido) ----
print('=== Adam Optimizer ===')
opt_adam = qml.AdamOptimizer(stepsize=0.1)
params_adam = qml_np.array([0.5, 0.1], requires_grad=True)
for step in range(50):
    params_adam, custo = opt_adam.step_and_cost(circuito_variacional, params_adam)
    if step % 10 == 0:
        print(f'  Step {step:2d}: custo={custo:.4f}, θ=[{params_adam[0]:.3f}, {params_adam[1]:.3f}]')

print(f'  Resultado final: ⟨Z⟩ = {circuito_variacional(params_adam):.4f} (mínimo = -1.0)')

=== Gradient Descent (SGD) ===
  Step  0: custo=0.8776, θ=[0.692, 0.100]
  Step 10: custo=-0.9945, θ=[3.079, 0.100]
  Step 20: custo=-1.0000, θ=[3.141, 0.100]
  Step 30: custo=-1.0000, θ=[3.142, 0.100]
  Step 40: custo=-1.0000, θ=[3.142, 0.100]
  Resultado final: ⟨Z⟩ = -1.0000 (mínimo = -1.0)

=== Adam Optimizer ===
  Step  0: custo=0.8776, θ=[0.600, 0.100]
  Step 10: custo=0.0581, θ=[1.616, 0.100]
  Step 20: custo=-0.8255, θ=[2.639, 0.100]
  Step 30: custo=-0.9845, θ=[3.363, 0.100]
  Step 40: custo=-0.9506, θ=[3.441, 0.100]
  Resultado final: ⟨Z⟩ = -0.9981 (mínimo = -1.0)


## Exemplo 4 — VQE: Encontrando a Energia do Estado Fundamental

**VQE (Variational Quantum Eigensolver)** é um dos algoritmos quânticos mais
importantes. Usa um circuito parametrizado para aproximar o **estado de menor
energia** de um Hamiltoniano — fundamental em química quântica.

**Hamiltoniano simples de 2 qubits:**
$$H = 0.5 \cdot Z_0 Z_1 - 0.3 \cdot X_0 - 0.3 \cdot X_1$$

O VQE minimiza `⟨ψ(θ)|H|ψ(θ)⟩` — o valor esperado da energia.
O mínimo corresponde ao ground state (energia mais baixa do sistema).

In [9]:
import pennylane as qml
import pennylane.numpy as qml_np
import numpy as np

# Hamiltoniano: H = 0.5*Z0Z1 - 0.3*X0 - 0.3*X1
coeffs    = [0.5, -0.3, -0.3]
observables = [
    qml.PauliZ(0) @ qml.PauliZ(1),  # Z tensor Z (correlação)
    qml.PauliX(0),
    qml.PauliX(1),
]
H = qml.Hamiltonian(coeffs, observables)
print('Hamiltoniano:')
print(H)

# Energia exata (numpy) para comparação
energia_exata = np.min(np.linalg.eigvalsh(qml.matrix(H, wire_order=[0, 1])))
print(f'\nEnergia exata do ground state: {energia_exata:.6f}')

# ---- Ansatz variacional ----
dev = qml.device("default.qubit", wires=2)

@qml.qnode(dev)
def vqe_ansatz(params):
    # Camada 1: rotações individuais
    qml.RY(params[0], wires=0)
    qml.RY(params[1], wires=1)
    # Entrelaçamento
    qml.CNOT(wires=[0, 1])
    # Camada 2: rotações pós-entrelaçamento
    qml.RY(params[2], wires=0)
    qml.RY(params[3], wires=1)
    return qml.expval(H)  # Energia esperada

# Treina o VQE
params = qml_np.array([0.1, 0.2, 0.3, 0.4], requires_grad=True)
opt = qml.AdamOptimizer(stepsize=0.05)

print('\nTreinamento do VQE:')
print(f'{'Step':>5} | {'Energia':>12} | {'Erro':>10}')
print('-' * 35)
for step in range(100):
    params, energia = opt.step_and_cost(vqe_ansatz, params)
    if step % 20 == 0 or step == 99:
        erro = abs(energia - energia_exata)
        print(f'{step:>5} | {energia:>12.6f} | {erro:>10.6f}')

print(f'\nEnergia VQE final:  {vqe_ansatz(params):.6f}')
print(f'Energia exata:      {energia_exata:.6f}')

Hamiltoniano:
0.5 * (Z(0) @ Z(1)) + -0.3 * X(0) + -0.3 * X(1)

Energia exata do ground state: -0.781025

Treinamento do VQE:
 Step |      Energia |       Erro
-----------------------------------
    0 |     0.137447 |   0.918472
   20 |    -0.720207 |   0.060818
   40 |    -0.738031 |   0.042994
   60 |    -0.771394 |   0.009631
   80 |    -0.780720 |   0.000305
   99 |    -0.780958 |   0.000067

Energia VQE final:  -0.780952
Energia exata:      -0.781025


## Exemplo 5 — PennyLane + PyTorch: Circuito como Camada Neural

O PennyLane permite inserir um circuito quântico **dentro** de uma rede
neural clássica como se fosse uma camada `nn.Linear`.

O backpropagation **passa pelo circuito quântico** automaticamente —
o gradiente flui de volta para as camadas anteriores.

### Arquitetura híbrida clássico-quântica
```
Input (4) → Linear(4→4) → [Circuito Quântico 4 qubits] → Linear(4→2) → Output
                               ↑ `TorchLayer`
                               (backprop atravessa)
```

In [10]:
import torch
import torch.nn as nn
import pennylane as qml
import numpy as np

# ---- Circuito quântico ----
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_layer(inputs, weights):
    # AngleEmbedding: codifica dados clássicos como ângulos de rotação
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    # BasicEntanglerLayers: camadas de rotações + CNOTs
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    # Retorna valor esperado de Z em cada qubit → 4 saídas
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# TorchLayer: transforma o qnode em nn.Module compatível
n_layers = 3
weight_shapes = {"weights": (n_layers, n_qubits)}
qlayer = qml.qnn.TorchLayer(quantum_layer, weight_shapes)

# ---- Modelo híbrido clássico-quântico ----
modelo = nn.Sequential(
    nn.Linear(4, 4),   # pré-processamento clássico
    nn.Tanh(),         # normaliza para [-1, +1] (necessário para AngleEmbedding)
    qlayer,            # ← CAMADA QUÂNTICA
    nn.Linear(4, 2),   # pós-processamento clássico
    nn.Softmax(dim=1), # probabilidades de classe
)

print('Arquitetura do modelo híbrido:')
print(modelo)
print(f'\nParâmetros totais: {sum(p.numel() for p in modelo.parameters())}')

# ---- Teste com dados aleatórios ----
batch_size = 8
X = torch.randn(batch_size, 4)
y = torch.randint(0, 2, (batch_size,))

saida = modelo(X)
print(f'\nEntrada shape: {X.shape}')
print(f'Saída shape:   {saida.shape}  (batch={batch_size}, classes=2)')
print(f'Saída (primeiros 3):  {saida[:3].detach().numpy().round(3)}')

Arquitetura do modelo híbrido:
Sequential(
  (0): Linear(in_features=4, out_features=4, bias=True)
  (1): Tanh()
  (2): <Quantum Torch Layer: func=quantum_layer>
  (3): Linear(in_features=4, out_features=2, bias=True)
  (4): Softmax(dim=1)
)

Parâmetros totais: 42

Entrada shape: torch.Size([8, 4])
Saída shape:   torch.Size([8, 2])  (batch=8, classes=2)
Saída (primeiros 3):  [[0.32  0.68 ]
 [0.321 0.679]
 [0.323 0.677]]


## Exemplo 6 — Treinamento Completo: Classificação Híbrida

Treinamento end-to-end de um modelo híbrido quântico-clássico
em um dataset de classificação binária simples.

O loop de treinamento é **idêntico** ao PyTorch clássico —
a única diferença é que dentro do `Sequential` existe uma camada quântica.

In [11]:
import torch
import torch.nn as nn
import pennylane as qml
import numpy as np

# ---- Dataset sintético: 2 classes separáveis ----
np.random.seed(42)
torch.manual_seed(42)
n_samples = 64
X_np = np.random.randn(n_samples, 4).astype(np.float32)
# Classe 0: soma dos features < 0 | Classe 1: soma dos features >= 0
y_np = (X_np.sum(axis=1) >= 0).astype(np.int64)

X_tensor = torch.tensor(X_np)
y_tensor = torch.tensor(y_np)

# ---- Modelo ----
n_qubits, n_layers = 4, 2
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def qnode(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

qlayer = qml.qnn.TorchLayer(qnode, {"weights": (n_layers, n_qubits)})

modelo = nn.Sequential(
    nn.Linear(4, 4), nn.Tanh(),
    qlayer,
    nn.Linear(4, 2)
)

# ---- Treinamento ----
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(modelo.parameters(), lr=0.01)

print(f'{'Época':>6} | {'Loss':>8} | {'Acurácia':>10}')
print('-' * 30)

for epoca in range(30):
    optimizer.zero_grad()
    saida = modelo(X_tensor)
    loss = loss_fn(saida, y_tensor)
    loss.backward()   # backprop atravessa a camada quântica!
    optimizer.step()

    if epoca % 5 == 0 or epoca == 29:
        pred = saida.argmax(dim=1)
        acc = (pred == y_tensor).float().mean().item()
        print(f'{epoca:>6} | {loss.item():>8.4f} | {acc:>9.1%}')

print('\nTreinamento concluído!')

 Época |     Loss |   Acurácia
------------------------------
     0 |   0.7279 |     39.1%
     5 |   0.6872 |     54.7%
    10 |   0.6487 |     68.8%
    15 |   0.6026 |     76.6%
    20 |   0.5505 |     84.4%
    25 |   0.4976 |     87.5%
    29 |   0.4566 |     90.6%

Treinamento concluído!


## Exemplo 7 — Data Re-uploading: Expressividade Máxima

**Data re-uploading** é uma técnica onde os dados de entrada são
codificados **múltiplas vezes**, intercalados com camadas parametrizadas.

Isso aumenta drasticamente a **capacidade expressiva** do circuito —
é o equivalente quântico de ter camadas profundas em uma rede neural.

```
Clássico:  x → [W₁] → ReLU → [W₂] → ReLU → y
Quântico:  x → [θ₁ + x] → [θ₂ + x] → [θ₃ + x] → ⟨Z⟩
                 ↑ re-upload dos dados em cada camada
```

In [12]:
import pennylane as qml
import pennylane.numpy as qml_np
import numpy as np

dev = qml.device("default.qubit", wires=1)

# 3 camadas de re-uploading — cada uma recodifica os dados
n_layers = 3

@qml.qnode(dev)
def data_reuploading(x, params):
    """Circuito de data re-uploading em 1 qubit.
    Cada camada: codifica x + aplica rotação aprendida.
    """
    for i in range(n_layers):
        # Codifica o dado x junto com o parâmetro aprendido
        qml.RY(params[i, 0] * x + params[i, 1], wires=0)
        qml.RZ(params[i, 2], wires=0)
    return qml.expval(qml.PauliZ(0))

# Visualiza o circuito
params_init = qml_np.random.uniform(0, np.pi, (n_layers, 3), requires_grad=True)
print('Circuito de Data Re-uploading (1 qubit, 3 camadas):')
print(qml.draw(data_reuploading)(0.5, params_init))

# ---- Aprende a função seno ----
# Objetivo: circuito aproxima sin(x) em [-π, π]
X_train = qml_np.linspace(-np.pi, np.pi, 20)
y_train = np.sin(X_train)

def custo(params):
    preds = qml_np.array([data_reuploading(x, params) for x in X_train])
    return qml_np.mean((preds - y_train) ** 2)

opt = qml.AdamOptimizer(stepsize=0.05)
params = params_init.copy()

print('\nAprendendo sin(x):')
for step in range(100):
    params, loss = opt.step_and_cost(custo, params)
    if step % 25 == 0:
        print(f'  Step {step:3d}: MSE = {loss:.6f}')

print('\nComparação (x, sin(x), predição):')
X_test = [-np.pi, -np.pi/2, 0, np.pi/2, np.pi]
for x in X_test:
    pred = data_reuploading(x, params)
    print(f'  x={x:>6.2f} | sin(x)={np.sin(x):>7.4f} | pred={pred:>7.4f}')

Circuito de Data Re-uploading (1 qubit, 3 camadas):
0: ──RY(1.26)──RZ(0.29)──RY(1.13)──RZ(3.08)──RY(0.33)──RZ(2.40)─┤  <Z>

Aprendendo sin(x):
  Step   0: MSE = 0.856957
  Step  25: MSE = 0.027810
  Step  50: MSE = 0.000815
  Step  75: MSE = 0.000095

Comparação (x, sin(x), predição):
  x= -3.14 | sin(x)=-0.0000 | pred= 0.0027
  x= -1.57 | sin(x)=-1.0000 | pred=-0.9968
  x=  0.00 | sin(x)= 0.0000 | pred= 0.0011
  x=  1.57 | sin(x)= 1.0000 | pred= 0.9974
  x=  3.14 | sin(x)= 0.0000 | pred=-0.0040


## Exemplo 8 — Backends: Agnosticismo de Hardware

Um dos pontos fortes do PennyLane: **você troca o device, o circuito não muda**.

| Device | Backend | Uso |
|--------|---------|-----|
| `default.qubit` | NumPy puro | Simulação exata (padrão) |
| `default.qubit.torch` | PyTorch | GPU acceleration |
| `default.qubit.jax` | JAX | JIT compilation |
| `lightning.qubit` | C++ (Kafka) | Simulação rápida em CPU |
| `lightning.gpu` | NVIDIA cuQuantum | Simulação em GPU |
| `qiskit.aer` | Qiskit Aer | Simulação com ruído realista |
| `braket.aws.qubit` | Amazon Braket | Hardware real (IonQ, Rigetti) |
| `ibmq` | IBM Quantum | Hardware IBM real |

Você pode rodar o **mesmo circuito** em todos esses backends.

In [13]:
import pennylane as qml
import numpy as np
import time

# O mesmo circuito roda em diferentes backends
def circuito_base(params, wires):
    """Circuito independente de backend."""
    for i in wires:
        qml.RY(params[i], wires=i)
    for i in range(len(wires) - 1):
        qml.CNOT(wires=[i, i+1])
    return [qml.expval(qml.PauliZ(i)) for i in wires]

n_qubits = 4
params = np.random.uniform(0, np.pi, n_qubits)

# ---- Backend 1: default.qubit (NumPy) ----
dev_numpy = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev_numpy)
def qnode_numpy(params):
    return circuito_base(params, range(n_qubits))

t0 = time.time()
resultado_numpy = qnode_numpy(params)
t_numpy = time.time() - t0
print(f'default.qubit (NumPy): {np.array(resultado_numpy).round(4)} | {t_numpy*1000:.2f}ms')

# ---- Desenha o circuito ----
print('\nCircuito:')
print(qml.draw(qnode_numpy)(params))

# ---- Informações do device ----
print(f'\nDevice: {dev_numpy.name}')
print(f'Método de diff: {qnode_numpy.diff_method}')
print(f'Interface:      {qnode_numpy.interface}')

default.qubit (NumPy): [-0.8216 -0.3815 -0.0422  0.0192] | 1.57ms

Circuito:
0: ──RY(2.53)─╭●───────┤  <Z>
1: ──RY(1.09)─╰X─╭●────┤  <Z>
2: ──RY(1.46)────╰X─╭●─┤  <Z>
3: ──RY(2.04)───────╰X─┤  <Z>

Device: default.qubit
Método de diff: best
Interface:      auto


## API Essencial do PennyLane

| Objeto / Função | Papel |
|-----------------|-------|
| `qml.device(name, wires=n)` | Seleciona backend de execução |
| `@qml.qnode(dev)` | Decora função como circuito quântico |
| `qml.RY(θ, wires=i)` | Porta de rotação parametrizada |
| `qml.expval(op)` | Retorna valor esperado do observable |
| `qml.probs(wires)` | Retorna vetor de probabilidades |
| `qml.grad(f)(params)` | Gradiente exato via parameter-shift |
| `qml.AdamOptimizer` | Otimizador Adam para circuitos |
| `qml.qnn.TorchLayer` | Integra qnode como camada PyTorch |
| `qml.draw(qnode)(params)` | Desenha o circuito em ASCII |